# 🏆 WAXAL ASR Challenge — Final Submission
**Author:** Bright Sikazwe
**Target Languages:** Lingala (lin), Shona (sna), Luganda (lug)
**Evaluation Metric:** 0.5 * WER + 0.5 * CER

## 📑 Zindi Code Review Documentation

### 1. Overview and Objectives
This notebook implements an automated, end-to-end Automatic Speech Recognition (ASR) fine-tuning pipeline. The objective is to maximize transcription accuracy for three low-resource African languages using OpenAI's `whisper-large-v3-turbo` architecture. To accommodate compute constraints while maximizing performance, the model is fine-tuned using **Low-Rank Adaptation (LoRA)**.

### 2. Environment & Reproducibility
* **Hardware:** Google Colab Pro (NVIDIA A100 / L4 GPU Premium class).
* **Randomness:** Global random seeds are pinned to `42` across Python, NumPy, and PyTorch to guarantee reproducible weight initialization and data shuffling.
* **Libraries:** Specific dependencies include `transformers`, `peft`, `datasets`, `librosa`, and `jiwer`. A targeted `torchao>=0.16.0` install is used to ensure compatibility with modern PEFT adapters.

### 3. Architecture & ETL Pipeline
**Extract:**
* Metadata (`Train.csv`, `Test.csv`, `SampleSubmission.csv`) is loaded from Google Drive.
* Audio arrays are extracted from the Hugging Face hub (`google/WaxalNLP`). **Design Choice:** To prevent disk overflow (OOM) from the 100GB+ unlabeled split, the pipeline uses a targeted `data_files` fetch to *only* pull the strictly necessary `train`, `validation`, and `test` parquet files.

**Transform:**
* **Data Cleansing:** Malformed rows in the raw CSVs (e.g., unescaped quotes causing column shifts) are dynamically dropped using `on_bad_lines='skip'`.
* **Zero Leakage Rule:** Text transcriptions are programmatically stripped from the `test` dataset object immediately upon extraction to guarantee zero label leakage.
* **Audio Resampling:** The PyTorch `WaxalDataset` class processes audio on-the-fly, using `librosa` to resample varying sample rates strictly to Whisper's required 16,000 Hz. Audio exceeding 30 seconds is truncated.
* **Language Mapping:** Luganda (`lug`) is not natively supported by Whisper's tokenizer. Following cross-lingual transfer best practices, it is mapped to Swahili (`swa`).

**Load & Model:**
* Data is passed via a custom PyTorch collator that dynamically tokenizes text and extracts log-mel spectrograms.
* The `whisper-large-v3-turbo` model is loaded in `FP16`.
* **PEFT/LoRA:** We inject adapter weights (Rank=32, Alpha=64) into the `q_proj`, `v_proj`, `k_proj`, and `out_proj` attention modules.

### 4. Inference & Performance Metrics
* **Inference:** The model utilizes batch generation (`max_new_tokens=200`).
* **Metrics:** Out-of-band evaluation calculates both raw and normalized Word Error Rate (WER) and Character Error Rate (CER) using the `jiwer` library. Text normalization removes punctuation and standardizes casing before scoring.

### 5. Error Handling & Maintenance
* **Fault Tolerance:** The `Seq2SeqTrainer` is configured to search the designated `work/` directory for recent checkpoints. If the compute instance disconnects, re-running the cell will automatically resume training from the last saved state.

**INSTALL LIBS**

In [ ]:
%%capture
!apt-get update -y -q
!apt-get install -y -q ffmpeg
!pip install -q -U transformers datasets peft accelerate jiwer soundfile librosa evaluate bitsandbytes "torchao>=0.16.0"

**MAIN CODE**

In [ ]:
# ==============================================================================
# 2. PIPELINE CONFIGURATION & REPRODUCIBILITY
# ==============================================================================
import os, random, numpy as np, torch
import pandas as pd
import datasets
from datasets import load_dataset, concatenate_datasets, Audio
import librosa
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
import jiwer
import re
import gc

# --- Reproducibility Anchor ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# --- Modeling Parameters ---
MODEL_ID   = "openai/whisper-large-v3-turbo"
USE_LORA   = True  # Utilizes PEFT to dramatically reduce memory footprint

LANGS = ["lin", "sna", "lug"]
# Design Choice: Luganda ("lug") mapped to "swahili" ("sw") for Whisper tokenizer compatibility
WHISPER_LANG = {"lin": "lingala", "sna": "shona", "lug": "swahili"}

# --- Storage Configuration ---
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/waxal"
os.makedirs(DRIVE_DIR, exist_ok=True)

ZINDI_DIR = DRIVE_DIR
WORK_DIR  = f"{DRIVE_DIR}/work"
os.makedirs(WORK_DIR, exist_ok=True)
DATASET_ID = "google/WaxalNLP"

# --- Training Hyperparameters ---
MAX_AUDIO_SECONDS = 30.0
NUM_EPOCHS        = 2
BATCH_SIZE        = 16
GRAD_ACCUM        = 4
LR                = 1e-4

# --- Hugging Face Authentication ---
from google.colab import userdata
from huggingface_hub import login
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

ACTIVE_GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Hardware Runtime: {ACTIVE_GPU}\nPipeline Configuration: Model={MODEL_ID} | LoRA={USE_LORA}")

# ==============================================================================
# 3. ETL PROCESS: EXTRACT & TRANSFORM
# ==============================================================================
print("\n[ETL Stage] Loading Zindi CSVs and cleaning anomalous entries...")

# Error Handling: on_bad_lines='skip' drops ~25 corrupted rows safely without crashing
train_df = pd.read_csv(f"{ZINDI_DIR}/Train.csv", dtype=str, keep_default_na=False, on_bad_lines='skip')
test_df  = pd.read_csv(f"{ZINDI_DIR}/Test.csv",  dtype=str)
sub_df   = pd.read_csv(f"{ZINDI_DIR}/SampleSubmission.csv", dtype=str)

train_df = train_df[train_df["language"].isin(LANGS)].copy()
train_df = train_df[train_df["transcription"].str.strip().ne("")]
test_df["language"] = test_df["ID"].str.split("_").str[0]

# Leakage Prevention: Ensure no target IDs exist in both splits
train_ids = set(train_df["id"])
test_ids = set(test_df["ID"])
assert not (train_ids & test_ids), "CRITICAL ERROR: Data contamination. Train and Test IDs overlap."

print("[ETL Stage] Downloading targeted WaxalNLP audio arrays from Hugging Face...")

def load_lang_dataset(lang: str):
    """
    Downloads language-specific configurations from the Hugging Face hub.

    Optimization: Uses targeted `data_files` regex to bypass the massive 'unlabeled'
    split, preventing Colab disk Out-Of-Memory (OOM) failures.
    Compliance: Strips text annotations from the test split immediately.
    """
    parts = []
    for split in ["train", "validation", "test"]:
        print(f"  -> Fetching {lang.upper()} / {split}...")
        try:
            file_pattern = f"data/ASR/{lang}/{lang}-{split}-*.parquet"
            ds = load_dataset(DATASET_ID, data_files={split: file_pattern}, split=split)
            ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        except Exception as e:
            print(f"     Skipped {split}: {e}")
            continue

        # Strip true labels from Test set instantly
        keep_columns = ["id", "audio"] + ([] if split == "test" else [c for c in ds.column_names if "transcript" in c.lower()])
        ds = ds.remove_columns([c for c in ds.column_names if c not in keep_columns])
        parts.append(ds)

    return concatenate_datasets(parts)

# Preemptively clear residual HF cache to ensure disk health
!rm -rf ~/.cache/huggingface/datasets/downloads
audio_ds = {lang: load_lang_dataset(lang) for lang in LANGS}

# ==============================================================================
# 4. DATA MODELING & PIPELINE PREPARATION
# ==============================================================================
print("\n[Data Modeling] Indexing datasets and loading Whisper architecture...")

# O(1) Index mapping for rapid lookup during dataloader yielding
id_to_idx = {}
for lang in LANGS:
    for i, _id in enumerate(audio_ds[lang]["id"]):
        id_to_idx[_id] = (lang, i)

train_split = train_df[train_df.original_split == "train"]
val_split   = train_df[train_df.original_split == "validation"]

def build_manifest(df: pd.DataFrame, with_text: bool = True) -> list:
    """Constructs a list of metadata dictionaries mapping Zindi IDs to HF audio indices."""
    recs = []
    trans_map = dict(zip(train_df["id"], train_df["transcription"])) if with_text else {}
    for _, r in df.iterrows():
        _id = r.get("id", r.get("ID"))
        if _id not in id_to_idx: continue
        lang, i = id_to_idx[_id]
        rec = {"id": _id, "lang": lang, "src_idx": i}
        if with_text: rec["text"] = trans_map[_id]
        recs.append(rec)
    return recs

train_recs = build_manifest(train_split)
val_recs   = build_manifest(val_split)
test_recs  = build_manifest(test_df, with_text=False)

# Initialize Processor and Model
processor = WhisperProcessor.from_pretrained(MODEL_ID, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if USE_LORA else torch.float32,
    low_cpu_mem_usage=True
)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

# Inject LoRA Adapters
if USE_LORA:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, target_modules=["q_proj", "v_proj", "k_proj", "out_proj"])
    model = get_peft_model(model, lora)
    print("\n✅ LoRA Adapters successfully injected. Model frozen, adapters trainable.")
else:
    model.gradient_checkpointing_enable()

class WaxalDataset(torch.utils.data.Dataset):
    """
    Custom PyTorch Dataset.
    Designed for memory efficiency: resamples and truncates audio arrays on-the-fly
    rather than loading the entire 38K corpus into RAM.
    """
    def __init__(self, recs, training=True):
        self.recs = recs
        self.training = training
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        row = audio_ds[r["lang"]][r["src_idx"]]
        audio_data = row["audio"]
        arr, sr = audio_data["array"], audio_data["sampling_rate"]

        # Enforce Whisper 16kHz sample rate requirement
        if sr != 16000:
            arr = librosa.resample(np.asarray(arr, dtype=np.float32), orig_sr=sr, target_sr=16000)

        arr = np.asarray(arr, dtype=np.float32)[: int(MAX_AUDIO_SECONDS * 16000)]
        output = {"array": arr, "lang": r["lang"], "id": r["id"]}
        if self.training:
            output["text"] = r["text"]
        return output

def make_collator(training=True):
    """Dynamically computes log-mel spectrogram features and tokenizes labels per batch."""
    def collate(batch):
        feats = processor.feature_extractor([b["array"] for b in batch], sampling_rate=16000, return_tensors="pt")
        out = {"input_features": feats.input_features}
        if training:
            labels = []
            for b in batch:
                processor.tokenizer.set_prefix_tokens(language=WHISPER_LANG[b["lang"]], task="transcribe")
                labels.append(processor.tokenizer(b["text"]).input_ids)
            maxlen = min(max(len(l) for l in labels), 448)
            lab = torch.full((len(labels), maxlen), -100, dtype=torch.long) # -100 ignores padding in loss calculation
            for i, l in enumerate(labels):
                l = l[:maxlen]
                lab[i, :len(l)] = torch.tensor(l)
            out["labels"] = lab
        return out
    return collate

train_ds = WaxalDataset(train_recs)
val_ds   = WaxalDataset(val_recs)

# ==============================================================================
# 5. MODEL TRAINING CONFIGURATION
# ==============================================================================
print("\n[Training Phase] Initializing Sequence-to-Sequence Trainer...")

args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK_DIR}/whisper-waxal-lora",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=300,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=50,
    save_steps=1000,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=make_collator(True)
)

# Fault Tolerance: Check for previous checkpoints to resume crashed runs
import glob
ckpts = glob.glob(f"{WORK_DIR}/whisper-waxal-lora/checkpoint-*")
resume_path = max(ckpts, key=os.path.getmtime) if ckpts else None

if resume_path:
    print(f"Resuming training from previous LoRA checkpoint: {resume_path}")
else:
    print("No checkpoints found. Starting fresh LoRA training run.")

trainer.train(resume_from_checkpoint=resume_path)

print("Training complete. Saving final model weights to Google Drive...")
trainer.save_model(f"{WORK_DIR}/final_lora")
processor.save_pretrained(f"{WORK_DIR}/final_lora")

# ==============================================================================
# 6. INFERENCE & EVALUATION (PERFORMANCE METRICS)
# ==============================================================================
print("\n[Evaluation Phase] Running inference on Validation and Test sets...")

model.eval()
model.half().cuda()

def text_normalize(text: str) -> str:
    """Standardizes target text parsing regex patterns to control metric calculations."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    return re.sub(r"\s+", " ", text)

@torch.no_grad()
def transcribe_inference_loop(manifest_recs, batch_size=16) -> dict:
    """Generates batch transcriptions using the fine-tuned Whisper model."""
    dataset_eval = WaxalDataset(manifest_recs, training=False)
    predictions = {}
    for start in range(0, len(dataset_eval), batch_size):
        batch = [dataset_eval[i] for i in range(start, min(start + batch_size, len(dataset_eval)))]
        feats = processor.feature_extractor([b["array"] for b in batch], sampling_rate=16000, return_tensors="pt")
        lang_token = WHISPER_LANG[batch[0]["lang"]]

        generated_ids = model.generate(feats.input_features.half().cuda(), language=lang_token, task="transcribe", max_new_tokens=200)
        decoded_strings = processor.batch_decode(generated_ids, skip_special_tokens=True)

        for b, transcribed_text in zip(batch, decoded_strings):
            predictions[b["id"]] = transcribed_text.strip()
        gc.collect()
    return predictions

# --- Out-of-Sample Performance Metric Reporting ---
print("\n--- Internal Cross-Validation Scores ---")
for lang in LANGS:
    lang_recs = [r for r in val_recs if r["lang"] == lang]
    preds = transcribe_inference_loop(lang_recs)
    references = [r["text"] for r in lang_recs]
    hypotheses = [preds[r["id"]] for r in lang_recs]

    raw_wer = jiwer.wer(references, hypotheses)
    raw_cer = jiwer.cer(references, hypotheses)
    norm_wer = jiwer.wer([text_normalize(x) for x in references], [text_normalize(x) for x in hypotheses])
    norm_cer = jiwer.cer([text_normalize(x) for x in references], [text_normalize(x) for x in hypotheses])

    print(f"[{lang.upper()}] Raw WER: {raw_wer:.4f} | Raw CER: {raw_cer:.4f}")
    print(f"[{lang.upper()}] Normalized WER: {norm_wer:.4f} | Normalized CER: {norm_cer:.4f}")

# --- Phase 1 Submission Delivery Construction ---
print("\nGenerating final Zindi leaderboard target array...")
all_test_predictions = {}
for lang in LANGS:
    lang_test_recs = [r for r in test_recs if r["lang"] == lang]
    all_test_predictions.update(transcribe_inference_loop(lang_test_recs))

final_submission = sub_df.copy()
final_submission["Target"] = final_submission["ID"].map(all_test_predictions)

# Post-processing edge case fallback validation (prevent Zindi scorer crashes)
assert final_submission["Target"].notna().all(), "CRITICAL ERROR: Omissions found within submission index map."
final_submission["Target"] = final_submission["Target"].str.strip().replace("", "a")

final_submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)
print(f"Process complete. Submission binary written cleanly to: {WORK_DIR}/submission.csv")

Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hardware Runtime: NVIDIA A100-SXM4-80GB
Pipeline Configuration: Model=openai/whisper-large-v3-turbo | LoRA=True

[ETL Stage] Loading Zindi CSVs and cleaning anomalous entries...
[ETL Stage] Downloading targeted WaxalNLP audio arrays from Hugging Face...
  -> Fetching LIN / train...


README.md:   0%|          | 0.00/29.2k [00:00<?, ?B/s]

data/ASR/lin/lin-train-00000.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

...

[SNA] Raw WER: 0.4621 | Raw CER: 0.1360
[SNA] Normalized WER: 0.3768 | Normalized CER: 0.1221


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[LUG] Raw WER: 0.4334 | Raw CER: 0.1574
[LUG] Normalized WER: 0.3720 | Normalized CER: 0.1458

Generating final Zindi leaderboard target array...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Process complete. Submission binary written cleanly to: /content/drive/MyDrive/waxal/work/submission.csv
